# 12.1 双曲型偏微分方程

双曲型方程描述信息沿特征传播。数值格式必须尊重传播方向和 CFL 条件，否则可能出现非物理振荡或不稳定增长。

In [ ]:
import math
import pathlib
import sys

import numpy as np

ROOT = pathlib.Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = next(parent for parent in pathlib.Path.cwd().parents if (parent / 'src').exists())
SRC = ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from py_sc import (
    lax_friedrichs_advection_1d,
    lax_wendroff_advection_1d,
    periodic_grid_1d,
    solve_wave_1d_dirichlet,
    upwind_advection_1d,
    upwind_advection_2d,
)

## 一维对流方程

常系数方程 $u_t+a u_x=0$ 的解析解是沿特征线平移：$u(x,t)=u_0(x-at)$。周期边界下，平滑正弦波可以用来比较数值耗散和色散。

In [ ]:
points = 80
x = periodic_grid_1d(0.0, 1.0, points)
dx = 1.0 / points
dt = 0.4 * dx
steps = 20
initial = np.sin(2.0 * math.pi * x)
exact = np.sin(2.0 * math.pi * ((x - steps * dt) % 1.0))

comparison = []
for solver in [upwind_advection_1d, lax_friedrichs_advection_1d, lax_wendroff_advection_1d]:
    result = solver(initial, velocity=1.0, dx=dx, dt=dt, steps=steps)
    error = np.linalg.norm(result.values[-1] - exact, ord=np.inf)
    amplitude = np.max(np.abs(result.values[-1]))
    comparison.append((result.method, result.courant, error, amplitude))
comparison

## 方波实验

不连续数据会放大格式差异。上风格式更耗散，Lax-Wendroff 在间断附近更容易产生色散振荡。

In [ ]:
square = np.where((x > 0.3) & (x < 0.6), 1.0, 0.0)
upwind_square = upwind_advection_1d(square, 1.0, dx, dt, steps)
lw_square = lax_wendroff_advection_1d(square, 1.0, dx, dt, steps)
{
    'upwind_minmax': (float(np.min(upwind_square.values[-1])), float(np.max(upwind_square.values[-1]))),
    'lax_wendroff_minmax': (float(np.min(lw_square.values[-1])), float(np.max(lw_square.values[-1]))),
    'mass_difference': float(np.sum(upwind_square.values[-1]) - np.sum(square)),
}

## 二维对流方程

二维常系数方程 $u_t+a u_x+b u_y=0$ 可以用直接上风格式离散。周期边界和单位 CFL 的单方向移动可以作为离散平移测试。

In [ ]:
initial2 = np.arange(12.0).reshape(3, 4)
adv2 = upwind_advection_2d(initial2, velocity_x=1.0, velocity_y=0.0, dx=0.5, dy=0.25, dt=0.5, steps=1)
adv2.values[-1], np.roll(initial2, 1, axis=0)

## 一维波动方程

波动方程 $u_{tt}=c^2u_{xx}$ 使用时间和空间中心差分。第一时间层由初始位移、初始速度和方程本身构造。

In [ ]:
wave_points = 101
wave_x = np.linspace(0.0, 1.0, wave_points)
wave_dx = wave_x[1] - wave_x[0]
wave_dt = 0.4 * wave_dx
wave_steps = 50
displacement = np.sin(math.pi * wave_x)
velocity = np.zeros_like(wave_x)
wave = solve_wave_1d_dirichlet(displacement, velocity, speed=1.0, dx=wave_dx, dt=wave_dt, steps=wave_steps)
wave_exact = np.cos(math.pi * wave.times[-1]) * np.sin(math.pi * wave_x)
np.linalg.norm(wave.values[-1] - wave_exact, ord=np.inf), wave.energy_history[-1] - wave.energy_history[0]

## 小结

* 双曲型 PDE 的离散必须匹配传播方向和 CFL 条件。
* 上风格式稳健但耗散，Lax-Friedrichs 耗散更明显，Lax-Wendroff 在平滑区域更准确但可能色散。
* 波动方程中心差分需要构造第一时间层，并可用解析模态和能量变化检查质量。